In [29]:
source(here::here("settings.R"))

In [30]:
# I/O
io$output.directory <- file.path(io$basedir,"ArchR")
dir.create(file.path(io$output.directory), showWarnings = FALSE)

setwd(io$output.directory)

In [31]:
opts = list()
# Options
opts$min.fragments <- 2500
opts$filterTSS.score <- 2 # May need to rerun with threshold at 2

# ArchR options
addArchRThreads(threads = 1) 

#important as rabbit chromosome dont have the 'chr' prefix
addArchRChrPrefix(chrPrefix = FALSE)

Setting default number of Parallel threads to 1.

ArchR is now disabling the requirement of chromosome prefix = 'chr'



In [45]:
genomeAnnotation = readRDS(file.path(io$basedir, 'genomeAnnotation.rds'))
geneAnnotation = readRDS(file.path(io$basedir, 'geneAnnotation.rds'))

In [46]:
fragment_files = list.files(file.path(io$basedir, 'data/'))

In [47]:
fragment_files

character(0)

In [48]:
#Create Arrow File for filtered samples, filtered using Signac's filter for ATAC
ArrowFiles <- createArrowFiles(
  inputFiles = file.path(io$basedir, 'data', fragment_files),
  sampleNames = paste0('rabbit_', strsplit(fragment_files,"_") %>% map_chr(1)),
  minTSS = opts$filterTSS.score, #Dont set this too high because you can always increase later
  minFrags = opts$min.fragments , 
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation
)

ERROR: Error in createArrowFiles(inputFiles = file.path(io$basedir, "data", fragment_files), : sampleNames must be equal length to inputFiles


In [ ]:
ArrowFiles = list.files(io$output.directory, pattern ='arrow')

In [ ]:
# Calculate doublet scores
doubScores <- addDoubletScores(
  input = ArrowFiles,
  k = 10, #Refers to how many cells near a "pseudo-doublet" to count.
  knnMethod = "UMAP", #Refers to the embedding to use for nearest neighbor search.
  LSIMethod = 1
)

In [ ]:
proj <- ArchRProject(
  ArrowFiles = ArrowFiles, 
  outputDirectory = "Project",
  copyArrows = TRUE, #This is recommened so that you maintain an unaltered copy for later usage.
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation
)

In [ ]:
proj <- filterDoublets(ArchRProj = proj)

In [ ]:
proj <- saveArchRProject(ArchRProj = proj)